# FDPIS — Notebook 7: Weather Integration (Layer 3 candidate)

**Run `fdpis_1_data_prep.ipynb` first.** This notebook does not modify notebook 2 or any
production artifact. Its model is saved as a **candidate** under `results/candidates/weather_b/`.

### The train/serve skew trap
Historically we know the *actual* weather at departure. In production, 3–6 hours out, we only
know a forecast or the latest observation. Training on departure-hour actuals is perfect
hindsight — a mild form of leakage. So two variants are built and both are reported:

| Variant | Weather used | Status |
|---|---|---|
| **A — at departure** | Origin obs at the scheduled departure hour, destination obs at the scheduled arrival hour | **Upper bound.** Equivalent to a perfect forecast. Not deployable. |
| **B — 3 h before** | Latest obs at least 3 h before scheduled departure, for origin *and* destination | **Honest number.** Genuinely available at prediction time. |

The A − B gap is the value of a perfect forecast. B is what the system would report.

### The second trap: timezones
METAR is UTC; BTS schedules are local airport time. Joining without conversion misaligns every
row by 4–10 h and nothing errors. Every METAR timestamp is converted to the airport's IANA zone,
and the join is then *proven* against BTS's own `WEATHER_DELAY` cause code before any model runs.

## 1 — Imports, paths, constants

In [1]:
import json, time, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
import joblib
import airportsdata

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

WX_DIR   = Path('data/weather')
RAW_DIR  = WX_DIR / 'raw'
CAND_DIR = Path('results/candidates/weather_b')
RAW_DIR.mkdir(parents=True, exist_ok=True)

KEYS = ['FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'DEST']
WX_START, WX_END = pd.Timestamp('2025-01-01', tz='UTC'), pd.Timestamp('2025-08-01', tz='UTC')

df = pd.read_parquet('data/processed.parquet')
assert not df.duplicated(KEYS).any(), 'flight key is not unique'
print(f'Loaded {len(df):,} flights, {df["FL_DATE"].min().date()} to {df["FL_DATE"].max().date()}')

Loaded 1,718,426 flights, 2025-01-01 to 2025-07-31


## 2 — Fetch METAR from the IEM ASOS archive (cached)

One request per station, one at a time, 1.5 s between requests. Every response is cached to
`data/weather/raw/{IATA}.csv`; a rerun never re-downloads. IEM indexes contiguous-US stations by
their 3-letter ID (ATL) but Alaska, Hawaii and the territories by ICAO (PHNL, PANC, PGUM), so an
empty IATA response is retried once with the ICAO code from `airportsdata`. The ID that worked is
recorded in `{IATA}.meta`.

In [2]:
BASE = 'https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py'
FIELDS = ['tmpf','vsby','sknt','gust','p01i','skyc1','skyc2','skyc3',
          'skyl1','skyl2','skyl3','wxcodes']
QUERY = ('&'.join(f'data={f}' for f in FIELDS)
         + '&year1=2025&month1=1&day1=1&year2=2025&month2=8&day2=1'
         + '&tz=Etc/UTC&format=onlycomma&latlon=no&missing=M&trace=T'
         + '&report_type=3&report_type=4')

def fetch(station_id):
    url = f'{BASE}?station={station_id}&{QUERY}'
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url, timeout=300) as r:
                return r.read().decode('utf-8')
        except Exception as e:
            print(f'  {station_id} attempt {attempt+1} failed: {e}')
            time.sleep(10 * (attempt + 1))
    raise RuntimeError(f'{station_id}: request failed 3 times')

n_rows = lambda text: max(len(text.strip().splitlines()) - 1, 0)

AP = airportsdata.load('IATA')
stations = sorted(set(df['ORIGIN']) | set(df['DEST']))
manifest = []
for s in stations:
    out, meta = RAW_DIR / f'{s}.csv', RAW_DIR / f'{s}.meta'
    if not (out.exists() and meta.exists()):
        text, used = fetch(s), s
        time.sleep(1.5)
        if n_rows(text) == 0 and AP.get(s, {}).get('icao'):
            used = AP[s]['icao']
            text = fetch(used)
            time.sleep(1.5)
        out.write_text(text, encoding='utf-8')
        meta.write_text(used if n_rows(text) else 'NONE')
    manifest.append({'iata': s, 'iem_id': meta.read_text().strip(),
                     'obs': n_rows(out.read_text(encoding='utf-8'))})
manifest = pd.DataFrame(manifest)
manifest.to_csv(WX_DIR / 'station_manifest.csv', index=False)

missing_st = manifest.loc[manifest['obs'] == 0, 'iata'].tolist()
via_icao = manifest[(manifest['obs'] > 0) & (manifest['iem_id'] != manifest['iata'])]
print(f'Stations requested : {len(manifest)}')
print(f'Stations returned  : {(manifest["obs"] > 0).sum()}  '
      f'({len(via_icao)} via ICAO fallback: {", ".join(via_icao["iata"] + "->" + via_icao["iem_id"])})')
print(f'Stations missing   : {len(missing_st)}  {missing_st}')
print(f'Observations       : {manifest["obs"].sum():,}  '
      f'(median {manifest.loc[manifest["obs"]>0, "obs"].median():,.0f} per station)')
sparse = manifest[(manifest['obs'] > 0) & (manifest['obs'] < 3000)]
if len(sparse):
    print(f'Sparse stations (<3,000 obs over 7 months): {sparse[["iata","obs"]].values.tolist()}')

Stations requested : 345
Stations returned  : 345  (42 via ICAO fallback: ADK->PADK, ADQ->PADQ, AKN->PAKN, ANC->PANC, AZA->KIWA, BET->PABE, BQN->TJBQ, BRW->PABR, CDV->PACV, CLD->KCRQ, DLG->PADL, FAI->PAFA, FCA->KGPI, GST->PAGS, GUF->KJKA, GUM->PGUM, HHH->KHXD, HNL->PHNL, ITO->PHTO, JNU->PAJN, KOA->PHKO, KTN->PAKT, LIH->PHLI, MQT->KSAW, OGG->PHOG, OME->PAOM, OTZ->PAOT, PBI->KDJT, PPG->NSTU, PSE->TJPS, PSG->PAPG, SCC->PASC, SCE->KUNV, SIT->PASI, SJU->TJSJ, SPN->PGSN, STT->TIST, STX->TISX, USA->KJQF, WRG->PAWG, YAK->PAYA, YUM->KNYL)
Stations missing   : 0  []
Observations       : 2,213,510  (median 6,256 per station)


## 3 — Parse, convert to local time, verify the conversion

`airportsdata` maps every BTS IATA code to an IANA zone. Conversion uses the zone database, so
DST is handled (all three study months have a fixed offset per zone; Arizona and Hawaii never
shift).

In [3]:
TZ = {s: AP[s]['tz'] for s in stations}
assert all(TZ.values()), 'airport without timezone'

frames = []
for s in stations:
    if s in missing_st:
        continue
    d = pd.read_csv(RAW_DIR / f'{s}.csv', na_values=['M'], keep_default_na=False,
                    dtype={'p01i': str, 'wxcodes': str, 'skyc1': str, 'skyc2': str, 'skyc3': str})
    d['station'] = s
    frames.append(d)
obs = pd.concat(frames, ignore_index=True)
obs['valid_utc'] = pd.to_datetime(obs['valid'], utc=True).astype('datetime64[ns, UTC]')
obs = obs[(obs['valid_utc'] >= WX_START) & (obs['valid_utc'] < WX_END)]
obs = obs.sort_values(['station', 'valid_utc']).drop_duplicates(['station', 'valid_utc'], keep='last')

obs['valid_local'] = pd.NaT
obs['tz'] = obs['station'].map(TZ)
local = pd.Series(index=obs.index, dtype='object')
for tz, idx in obs.groupby('tz').groups.items():
    local.loc[idx] = obs.loc[idx, 'valid_utc'].dt.tz_convert(tz).dt.tz_localize(None)
obs['valid_local'] = pd.to_datetime(local).astype('datetime64[ns]')
print(f'{len(obs):,} observations across {obs["station"].nunique()} stations')

print('\nTimezone verification (same instant, five zones):')
probe = pd.Timestamp('2025-07-15 18:00', tz='UTC')
rows = []
for s in ['ATL', 'ORD', 'DEN', 'PHX', 'LAX', 'HNL']:
    o = obs[(obs['station'] == s) & (obs['valid_utc'] >= probe)].iloc[0]
    rows.append({'station': s, 'tz': o['tz'], 'utc': o['valid_utc'].strftime('%Y-%m-%d %H:%M'),
                 'local': o['valid_local'].strftime('%Y-%m-%d %H:%M'),
                 'offset_h': (o['valid_local'] - o['valid_utc'].tz_localize(None)).total_seconds() / 3600})
tz_sample = pd.DataFrame(rows)
print(tz_sample.to_string(index=False))
expected = {'ATL': -4, 'ORD': -5, 'DEN': -6, 'PHX': -7, 'LAX': -7, 'HNL': -10}   # July (DST)
assert all(tz_sample.set_index('station')['offset_h'][k] == v for k, v in expected.items()), 'offset mismatch'
jan = obs[(obs['station'] == 'ATL') & (obs['valid_utc'].dt.month == 1)].iloc[0]
jan_off = (jan['valid_local'] - jan['valid_utc'].tz_localize(None)).total_seconds() / 3600
print(f"\nDST check: ATL in January offset = {jan_off:+.0f} h (EST, expect -5)")
assert jan_off == -5, 'DST not applied'
print('All offsets match the expected zone and DST state.')

2,213,152 observations across 345 stations

Timezone verification (same instant, five zones):
station                  tz              utc            local  offset_h
    ATL    America/New_York 2025-07-15 18:52 2025-07-15 14:52      -4.0
    ORD     America/Chicago 2025-07-15 18:51 2025-07-15 13:51      -5.0
    DEN      America/Denver 2025-07-15 18:35 2025-07-15 12:35      -6.0
    PHX     America/Phoenix 2025-07-15 18:51 2025-07-15 11:51      -7.0
    LAX America/Los_Angeles 2025-07-15 18:53 2025-07-15 11:53      -7.0
    HNL    Pacific/Honolulu 2025-07-15 18:53 2025-07-15 08:53     -10.0



DST check: ATL in January offset = -5 h (EST, expect -5)
All offsets match the expected zone and DST state.


## 4 — Per-observation weather features

* `WX_VISIBILITY` statute miles, capped at 10
* `WX_GUST_KT` 0 when no gust is reported (METAR omits the group when there is none)
* `WX_PRECIP_IN` hourly precip; trace (`T`) = 0.0001 in
* `WX_CEILING_FT` lowest BKN, OVC or VV layer (VV, vertical visibility, is how METAR reports an
  indefinite ceiling in fog — it is a ceiling under FAA rules); 10,000 if no such layer; capped
  at 10,000; NaN if the sky group is missing entirely
* `WX_IS_THUNDER/SNOW/FOG` substring match on present weather (`TS` also catches `TSRA`, `VCTS`)
* `WX_IS_IFR` visibility < 3 mi **or** ceiling < 1,000 ft

In [4]:
WX = ['VISIBILITY','WIND_KT','GUST_KT','PRECIP_IN','CEILING_FT','TEMP_F',
      'IS_THUNDER','IS_SNOW','IS_FOG','IS_IFR']

o = obs
f = pd.DataFrame({'station': o['station'], 'valid_utc': o['valid_utc']})
f['VISIBILITY'] = o['vsby'].clip(upper=10)
f['WIND_KT']    = o['sknt']
f['GUST_KT']    = o['gust'].fillna(0)
f['PRECIP_IN']  = pd.to_numeric(o['p01i'].replace('T', '0.0001'), errors='coerce')
f['TEMP_F']     = o['tmpf']

ceil = pd.Series(np.inf, index=o.index)
sky_any = pd.Series(False, index=o.index)
for i in (1, 2, 3):
    c, h = o[f'skyc{i}'].fillna('').str.strip(), o[f'skyl{i}']
    sky_any |= c.ne('')
    is_ceiling = c.isin(['BKN', 'OVC', 'VV'])
    ceil = np.where(is_ceiling & h.notna(), np.minimum(ceil, h), ceil)
ceil = pd.Series(ceil, index=o.index).replace(np.inf, 10000).clip(upper=10000)
f['CEILING_FT'] = ceil.where(sky_any)

wx = o['wxcodes'].fillna('')
f['IS_THUNDER'] = wx.str.contains('TS').astype(float)
f['IS_SNOW']    = wx.str.contains('SN').astype(float)
f['IS_FOG']     = wx.str.contains('FG').astype(float)
ifr = (f['VISIBILITY'] < 3) | (f['CEILING_FT'] < 1000)
f['IS_IFR'] = ifr.astype(float).where(f['VISIBILITY'].notna() | f['CEILING_FT'].notna())

print(f[WX].describe().T[['count','mean','min','max']].round(3).to_string())

                count      mean   min      max
VISIBILITY  2206875.0     8.753   0.0     10.0
WIND_KT     2199583.0     7.472   0.0    189.0
GUST_KT     2213152.0     4.035   0.0    231.0
PRECIP_IN   2211034.0     0.006   0.0      4.2
CEILING_FT  2204965.0  7066.755   0.0  10000.0
TEMP_F      2205583.0    54.939 -38.0    136.4
IS_THUNDER  2213152.0     0.024   0.0      1.0
IS_SNOW     2213152.0     0.050   0.0      1.0
IS_FOG      2213152.0     0.015   0.0      1.0
IS_IFR      2210474.0     0.133   0.0      1.0


## 5 — Hourly tables, one per variant

Both tables sit on a complete hourly UTC grid per station (all US zones have whole-hour offsets,
so a UTC hour is exactly one local hour).

* **Table A (nearest):** observation closest to the top of the hour, within ±30 min.
* **Table B (as-of):** latest observation *at or before* the top of the hour, within 59 min.
  Variant B looks up the hour `floor(scheduled departure) − 3 h`, so the observation it uses is
  always at least 3 h before departure. This is asserted per flight below rather than assumed.

Then missing hours are forward-filled up to 2 hours within the station, and left NaN beyond that.

In [5]:
def hourly(feats, direction, tolerance):
    out = []
    for s, g in feats.groupby('station', sort=False):
        grid = pd.DataFrame({'hour_utc': pd.date_range(WX_START, WX_END, freq='h', inclusive='left')
                             .astype('datetime64[ns, UTC]')})
        g = g.sort_values('valid_utc')
        m = pd.merge_asof(grid, g.drop(columns='station').rename(columns={'valid_utc': 'obs_utc'}),
                          left_on='hour_utc', right_on='obs_utc',
                          direction=direction, tolerance=tolerance)
        m[WX + ['obs_utc']] = m[WX + ['obs_utc']].ffill(limit=2)
        m['station'] = s
        out.append(m)
    t = pd.concat(out, ignore_index=True)
    t['hour_local'] = pd.NaT
    loc = pd.Series(index=t.index, dtype='object')
    for tz, idx in t.groupby(t['station'].map(TZ)).groups.items():
        loc.loc[idx] = t.loc[idx, 'hour_utc'].dt.tz_convert(tz).dt.tz_localize(None)
    t['hour_local'] = pd.to_datetime(loc).astype('datetime64[ns]')
    return t

tab_A = hourly(f, 'nearest',  pd.Timedelta(minutes=30))
tab_B = hourly(f, 'backward', pd.Timedelta(minutes=59))
for nm, t in [('A', tab_A), ('B', tab_B)]:
    print(f'Table {nm}: {len(t):,} station-hours, '
          f'{100*t["VISIBILITY"].isna().mean():.1f}% without a visibility value after ffill')

Table A: 1,755,360 station-hours, 1.0% without a visibility value after ffill
Table B: 1,755,360 station-hours, 1.0% without a visibility value after ffill


## 6 — Join to flights

| | Origin | Destination |
|---|---|---|
| **A** | local date + scheduled departure hour | local arrival date + scheduled arrival hour (next day when arrival < departure, notebook 1's rule) |
| **B** | `floor(dep_UTC) − 3 h` | **same instant** `floor(dep_UTC) − 3 h` at the destination |

B is keyed on UTC because origin and destination are often in different zones: "3 hours before
departure" is one instant, not one local clock reading.

In [6]:
fl = df[KEYS + ['CRS_DEP_TIME', 'CRS_ARR_TIME']].copy()
dep_h = fl['CRS_DEP_TIME'] // 100                       # 2400 -> 24 -> next day 00:00
arr_h = fl['CRS_ARR_TIME'] // 100
fl['dep_local_hour'] = (fl['FL_DATE'] + pd.to_timedelta(dep_h, unit='h')).astype('datetime64[ns]')
fl['arr_local_hour'] = (fl['FL_DATE'] + pd.to_timedelta((fl['CRS_ARR_TIME'] < fl['CRS_DEP_TIME']).astype(int), unit='D')
                        + pd.to_timedelta(arr_h, unit='h')).astype('datetime64[ns]')

dep_local_exact = (fl['FL_DATE'] + pd.to_timedelta(dep_h * 60 + fl['CRS_DEP_TIME'] % 100, unit='m')).astype('datetime64[ns]')
dep_utc = pd.Series(index=fl.index, dtype='datetime64[ns, UTC]')
for tz, idx in fl.groupby(fl['ORIGIN'].map(TZ)).groups.items():
    dep_utc.loc[idx] = (dep_local_exact.loc[idx].dt.tz_localize(tz, nonexistent='shift_forward',
                                                                ambiguous='NaT').dt.tz_convert('UTC'))
fl['dep_utc'] = dep_utc
fl['b_hour_utc'] = fl['dep_utc'].dt.floor('h') - pd.Timedelta(hours=3)

def attach(fl, tab, airport_col, left_key, right_key, prefix):
    t = tab[['station', right_key, 'obs_utc'] + WX].rename(
        columns={'station': airport_col, right_key: left_key, 'obs_utc': f'{prefix}OBS_UTC',
                 **{w: f'{prefix}{w}' for w in WX}})
    out = fl.merge(t, on=[airport_col, left_key], how='left', validate='many_to_one')
    assert len(out) == len(fl)
    return out

wA = attach(fl, tab_A, 'ORIGIN', 'dep_local_hour', 'hour_local', 'ORIG_WX_')
wA = attach(wA, tab_A, 'DEST',   'arr_local_hour', 'hour_local', 'DEST_WX_')
wB = attach(fl, tab_B, 'ORIGIN', 'b_hour_utc', 'hour_utc', 'ORIG_WX_')
wB = attach(wB, tab_B, 'DEST',   'b_hour_utc', 'hour_utc', 'DEST_WX_')

# The honesty guarantee for B, checked on every flight rather than assumed.
for side in ['ORIG', 'DEST']:
    lead = (wB['dep_utc'] - wB[f'{side}_WX_OBS_UTC']).dropna()
    assert (lead >= pd.Timedelta(hours=3)).all(), f'{side}: B uses an observation < 3h before departure'
    print(f'Variant B {side}: every observation is >= 3h before departure '
          f'(min lead {lead.min()}, median {lead.median()})')

ORIG_WX = [f'ORIG_WX_{w}' for w in WX]
DEST_WX = [f'DEST_WX_{w}' for w in WX]
WX_FEATURES = ORIG_WX + DEST_WX

def missing_report(w, name):
    r = {'variant': name,
         'origin_no_weather_%': round(100 * w['ORIG_WX_OBS_UTC'].isna().mean(), 2),
         'dest_no_weather_%':   round(100 * w['DEST_WX_OBS_UTC'].isna().mean(), 2)}
    for c in WX_FEATURES:
        r[c] = round(100 * w[c].isna().mean(), 2)
    return r

miss = pd.DataFrame([missing_report(wA, 'A'), missing_report(wB, 'B')]).set_index('variant').T
print('\nMissing-data rate, % of all flights (after <=2 h forward-fill):')
print(miss.to_string())
miss.to_csv(WX_DIR / 'missing_rates.csv')

wf = df[KEYS].copy()
for w, sfx in [(wA, '_A'), (wB, '_B')]:
    for c in WX_FEATURES:
        wf[c + sfx] = w[c].values
wf.to_parquet(WX_DIR / 'weather_features.parquet', index=False)
print(f'\nSaved {WX_DIR / "weather_features.parquet"}: {wf.shape[0]:,} rows x {wf.shape[1]} cols '
      f'(keys {KEYS}; suffix _A / _B per variant)')

Variant B ORIG: every observation is >= 3h before departure (min lead 0 days 03:00:00, median 0 days 03:34:00)
Variant B DEST: every observation is >= 3h before departure (min lead 0 days 03:00:00, median 0 days 03:34:00)



Missing-data rate, % of all flights (after <=2 h forward-fill):
variant                 A     B
origin_no_weather_%  0.27  0.15
dest_no_weather_%    0.42  0.14
ORIG_WX_VISIBILITY   0.28  0.17
ORIG_WX_WIND_KT      0.28  0.17
ORIG_WX_GUST_KT      0.27  0.15
ORIG_WX_PRECIP_IN    0.27  0.15
ORIG_WX_CEILING_FT   0.28  0.16
ORIG_WX_TEMP_F       0.28  0.17
ORIG_WX_IS_THUNDER   0.27  0.15
ORIG_WX_IS_SNOW      0.27  0.15
ORIG_WX_IS_FOG       0.27  0.15
ORIG_WX_IS_IFR       0.27  0.15
DEST_WX_VISIBILITY   0.42  0.15
DEST_WX_WIND_KT      0.43  0.15
DEST_WX_GUST_KT      0.42  0.14
DEST_WX_PRECIP_IN    0.42  0.14
DEST_WX_CEILING_FT   0.43  0.15
DEST_WX_TEMP_F       0.43  0.15
DEST_WX_IS_THUNDER   0.42  0.14
DEST_WX_IS_SNOW      0.42  0.14
DEST_WX_IS_FOG       0.42  0.14
DEST_WX_IS_IFR       0.42  0.14



Saved data\weather\weather_features.parquet: 1,718,426 rows x 45 cols (keys ['FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'DEST']; suffix _A / _B per variant)


## 7 — Sanity check: prove the join is correct

BTS's `WEATHER_DELAY` cause code is an independent answer key. Flights with `WEATHER_DELAY > 0`
must show visibly worse origin weather (Variant A) than flights with `WEATHER_DELAY == 0`.
If they do not, the join is wrong — almost certainly timezones — and the notebook **stops**.

A second, stronger check: shift the origin join key by −8…+8 hours. If the alignment is right,
the separation between the two groups should peak at or near zero shift (weather persists, so
±1–2 h is expected; a timezone bug would put the peak 4–10 h away). A naive
"treat UTC as local" join is the −4…−10 h region of that curve.

In [7]:
has_cause = df['WEATHER_DELAY'].notna()
wd = (df['WEATHER_DELAY'] > 0)
g_wd, g_no = has_cause & wd, has_cause & ~wd

checks = []
for c in ['ORIG_WX_IS_IFR', 'ORIG_WX_IS_THUNDER', 'ORIG_WX_VISIBILITY',
          'ORIG_WX_CEILING_FT', 'ORIG_WX_GUST_KT', 'ORIG_WX_PRECIP_IN']:
    a, b = wA.loc[g_wd.values, c].mean(), wA.loc[g_no.values, c].mean()
    checks.append({'feature': c, 'WEATHER_DELAY>0': round(a, 4), 'WEATHER_DELAY=0': round(b, 4),
                   'ratio': round(a / b, 2) if b else np.nan})
sanity = pd.DataFrame(checks)
print(f'Flights with cause codes: {has_cause.sum():,}  |  WEATHER_DELAY>0: {g_wd.sum():,}  '
      f'|  WEATHER_DELAY=0: {g_no.sum():,}\n')
print(sanity.to_string(index=False))

s = sanity.set_index('feature')
passed = (s.loc['ORIG_WX_IS_IFR', 'WEATHER_DELAY>0'] > s.loc['ORIG_WX_IS_IFR', 'WEATHER_DELAY=0']
          and s.loc['ORIG_WX_IS_THUNDER', 'WEATHER_DELAY>0'] > s.loc['ORIG_WX_IS_THUNDER', 'WEATHER_DELAY=0']
          and s.loc['ORIG_WX_VISIBILITY', 'WEATHER_DELAY>0'] < s.loc['ORIG_WX_VISIBILITY', 'WEATHER_DELAY=0'])

# Shift scan on the thunder rate ratio.
tA = tab_A[['station', 'hour_local', 'IS_THUNDER', 'IS_IFR']].rename(columns={'station': 'ORIGIN'})
base_keys = fl.loc[has_cause.values, ['ORIGIN', 'dep_local_hour']].copy()
y_wd = wd[has_cause].values
scan = []
for k in range(-8, 9):
    q = base_keys.assign(hour_local=base_keys['dep_local_hour'] + pd.Timedelta(hours=k))
    m = q.merge(tA, on=['ORIGIN', 'hour_local'], how='left')
    th, ifr_ = m['IS_THUNDER'].values, m['IS_IFR'].values
    scan.append({'shift_h': k,
                 'thunder_ratio': round(np.nanmean(th[y_wd]) / np.nanmean(th[~y_wd]), 2),
                 'ifr_ratio': round(np.nanmean(ifr_[y_wd]) / np.nanmean(ifr_[~y_wd]), 2)})
scan = pd.DataFrame(scan)
print('\nKey-shift scan (ratio of rate in weather-delayed vs other flights):')
print(scan.to_string(index=False))
peak = int(scan.loc[scan['thunder_ratio'].idxmax(), 'shift_h'])
print(f'\nThunder separation peaks at shift {peak:+d} h')

# A timezone bug shows up as a peak 4-10 h away from zero; 1-2 h either side is weather persistence.
if not passed or abs(peak) >= 4:
    raise AssertionError('SANITY CHECK FAILED - weather-delayed flights do not show worse weather '
                         'at the joined hour. The join is wrong. Stopping.')
print('SANITY CHECK PASSED - weather-delayed flights show clearly worse weather at the joined hour, '
      'and the separation peaks at the correct alignment.')

Flights with cause codes: 388,746  |  WEATHER_DELAY>0: 29,050  |  WEATHER_DELAY=0: 359,696

           feature  WEATHER_DELAY>0  WEATHER_DELAY=0  ratio
    ORIG_WX_IS_IFR           0.1384           0.0559   2.48
ORIG_WX_IS_THUNDER           0.2100           0.0199  10.54
ORIG_WX_VISIBILITY           8.3940           9.4212   0.89
ORIG_WX_CEILING_FT        6877.0142        8099.7295   0.85
   ORIG_WX_GUST_KT           5.6513           4.8709   1.16
 ORIG_WX_PRECIP_IN           0.0330           0.0041   7.97



Key-shift scan (ratio of rate in weather-delayed vs other flights):
 shift_h  thunder_ratio  ifr_ratio
      -8           2.62       1.53
      -7           2.89       1.56
      -6           3.18       1.59
      -5           3.32       1.64
      -4           3.31       1.73
      -3           3.74       1.87
      -2           4.88       2.01
      -1           6.67       2.17
       0          10.54       2.48
       1          10.02       2.53
       2           6.52       2.32
       3           4.20       2.22
       4           3.42       2.07
       5           2.46       2.03
       6           2.13       1.92
       7           1.49       1.84
       8           1.38       1.83

Thunder separation peaks at shift +0 h
SANITY CHECK PASSED - weather-delayed flights show clearly worse weather at the joined hour, and the separation peaks at the correct alignment.


## 8 — Reproduce the production setup exactly (notebook 2)

Same primary filter, same date-fraction split, same categorical dtypes from train, same
`scale_pos_weight`, same three blend components with the same hyperparameters and
`random_state=42`. The target encodings are not recomputed: production does not use them, and
they do not touch any of the 32 features. Weather columns are merged on the flight key with a
left join that preserves row order, so the baseline sees exactly notebook 2's matrices.

In [8]:
TARGET = 'DEP_DEL15'
data = df.copy()
for c in WX_FEATURES:
    data[c + '_A'] = wA[c].values
    data[c + '_B'] = wB[c].values

primary = data[data['LATE_AIRCRAFT_DELAY'].fillna(0) == 0].copy()
primary = primary.dropna(subset=[TARGET]).copy()
primary[TARGET] = primary[TARGET].astype(int)

dates = np.sort(primary['FL_DATE'].unique())
n = len(dates)
train_end, val_end = dates[int(n * 0.70)], dates[int(n * 0.85)]
train = primary[primary['FL_DATE'] <= train_end].copy()
val   = primary[(primary['FL_DATE'] > train_end) & (primary['FL_DATE'] <= val_end)].copy()
test  = primary[primary['FL_DATE'] > val_end].copy()
for nm, d in [('Train', train), ('Val', val), ('Test', test)]:
    print(f'{nm:<6} {len(d):>9,} | {d["FL_DATE"].min().date()} to {d["FL_DATE"].max().date()} '
          f'| positive {100*d[TARGET].mean():.1f}%')

def add_cyclical(d):
    d = d.copy()
    d['DEP_HOUR_SIN'] = np.sin(2*np.pi*d['DEP_HOUR']/24)
    d['DEP_HOUR_COS'] = np.cos(2*np.pi*d['DEP_HOUR']/24)
    d['ARR_HOUR_SIN'] = np.sin(2*np.pi*d['ARR_HOUR']/24)
    d['ARR_HOUR_COS'] = np.cos(2*np.pi*d['ARR_HOUR']/24)
    d['DOW_SIN'] = np.sin(2*np.pi*d['DAY_OF_WEEK']/7)
    d['DOW_COS'] = np.cos(2*np.pi*d['DAY_OF_WEEK']/7)
    return d
train, val, test = add_cyclical(train), add_cyclical(val), add_cyclical(test)

GROUPS = {
    'A_schedule':    ['CRS_DEP_TIME','CRS_ARR_TIME','DEP_HOUR','ARR_HOUR',
                      'CRS_ELAPSED_TIME','DISTANCE','SPEED_PROXY','DAY_OF_WEEK','IS_WEEKEND'],
    'B_cyclical':    ['DEP_HOUR_SIN','DEP_HOUR_COS','ARR_HOUR_SIN','ARR_HOUR_COS','DOW_SIN','DOW_COS'],
    'C_categorical': ['OP_UNIQUE_CARRIER','ORIGIN','DEST'],
    'E_congestion':  ['ORIGIN_HOUR_DEPARTURES','DEST_HOUR_ARRIVALS','ORIGIN_DAY_DEPARTURES',
                      'CARRIER_ORIGIN_FLIGHTS','CARRIER_ORIGIN_SHARE'],
    'F_rotation':    ['LEG_NUM','TAIL_LEGS_TODAY','LEG_FRACTION','IS_LAST_LEG',
                      'MINUTES_INTO_TAIL_DAY','SCHEDULED_BUFFER_MIN','AVAILABLE_SLACK',
                      'IS_TIGHT_TURNAROUND','LONG_GROUND_TIME'],
}
PROD_FEATURES = [f for g in GROUPS.values() for f in g]
assert len(PROD_FEATURES) == 32
CAT_COLS = ['OP_UNIQUE_CARRIER','ORIGIN','DEST']
cat_types = {c: pd.CategoricalDtype(categories=sorted(train[c].dropna().unique())) for c in CAT_COLS}

y_train, y_val, y_test = train[TARGET], val[TARGET], test[TARGET]
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

def prep(d, feats, rename=None):
    X = d[feats].copy()
    if rename:
        X = X.rename(columns=rename)
    for c in CAT_COLS:
        if c in X.columns:
            X[c] = X[c].astype(cat_types[c])
    return X

def to_numeric(X):
    Xn = X.copy()
    for c in CAT_COLS:
        Xn[c] = Xn[c].cat.codes
    return Xn

def precision_at_k(y_true, prob, k_pct):
    k = max(int(len(prob) * k_pct / 100), 1)
    return np.asarray(y_true)[np.argsort(-prob)[:k]].mean()

def weighted_carrier_p1(prob):
    # Notebook 2 cell 34: carriers with >= 2,000 test flights, top 1% within each, weighted by volume.
    t = test[['OP_UNIQUE_CARRIER', TARGET]].reset_index(drop=True).assign(prob=prob)
    rows = []
    for carr, g in t.groupby('OP_UNIQUE_CARRIER'):
        if len(g) < 2000:
            continue
        k = max(int(len(g) * 0.01), 1)
        rows.append((len(g), g.nlargest(k, 'prob')[TARGET].mean()))
    r = np.array(rows)
    return (r[:, 0] * r[:, 1]).sum() / r[:, 0].sum()

def evaluate(name, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {'config': name,
            'auc':    round(roc_auc_score(y_test, prob), 4),
            'pr_auc': round(average_precision_score(y_test, prob), 4),
            'f1':     round(f1_score(y_test, pred), 3),
            'precision': round(precision_score(y_test, pred, zero_division=0), 3),
            'recall': round(recall_score(y_test, pred), 3),
            'p@0.5%': round(precision_at_k(y_test, prob, 0.5), 3),
            'p@1%':   round(precision_at_k(y_test, prob, 1), 3),
            'p@5%':   round(precision_at_k(y_test, prob, 5), 3),
            'p@10%':  round(precision_at_k(y_test, prob, 10), 3),
            'carrier_wtd_p@1%': round(weighted_carrier_p1(prob), 3)}

def run_blend(feats, rename=None, tag=''):
    t0 = time.time()
    Xtr, Xv, Xte = prep(train, feats, rename), prep(val, feats, rename), prep(test, feats, rename)
    xgb_m = xgb.XGBClassifier(
        n_estimators=800, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        scale_pos_weight=scale_pos_weight, enable_categorical=True, tree_method='hist',
        eval_metric='auc', early_stopping_rounds=50, random_state=42, n_jobs=-1)
    xgb_m.fit(Xtr, y_train, eval_set=[(Xv, y_val)], verbose=False)
    hgb = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_depth=7,
                                         class_weight='balanced', random_state=42)
    hgb.fit(to_numeric(Xtr), y_train)
    lgbm = lgb.LGBMClassifier(n_estimators=800, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight,
        random_state=42, n_jobs=-1, verbose=-1)
    lgbm.fit(to_numeric(Xtr), y_train)
    probs = {'xgb': xgb_m.predict_proba(Xte)[:, 1],
             'hgb': hgb.predict_proba(to_numeric(Xte))[:, 1],
             'lgb': lgbm.predict_proba(to_numeric(Xte))[:, 1]}
    blend = np.mean([probs['xgb'], probs['hgb'], probs['lgb']], axis=0)
    print(f'{tag:<26} {len(feats)} features | xgb best iter {xgb_m.best_iteration} | '
          f'{time.time()-t0:.0f}s')
    return {'blend': blend, 'probs': probs, 'models': {'xgb': xgb_m, 'hgb': hgb, 'lgb': lgbm},
            'features': list(Xtr.columns)}

Train  1,067,167 | 2025-01-01 to 2025-07-04 | positive 11.3%
Val      232,583 | 2025-07-05 to 2025-07-18 | positive 17.0%
Test     223,101 | 2025-07-19 to 2025-07-31 | positive 15.6%


scale_pos_weight: 7.88


### 8.1 — Baseline must reproduce AUC 0.6924 and p@1% 0.781 before anything else runs

In [9]:
runs = {}
runs['BASELINE'] = run_blend(PROD_FEATURES, tag='BASELINE')
base_eval = evaluate('BASELINE (32 production)', runs['BASELINE']['blend'])
print(base_eval)

TOL = 0.002
for m, exp in {'auc': 0.6924, 'p@1%': 0.781}.items():
    if abs(base_eval[m] - exp) > TOL:
        raise AssertionError(f'Baseline does not reproduce production: {m} = {base_eval[m]} vs {exp}. Stopping.')
print(f'Baseline reproduces production (AUC {base_eval["auc"]}, p@1% {base_eval["p@1%"]}) within +/-{TOL}.')

BASELINE                   32 features | xgb best iter 68 | 76s


{'config': 'BASELINE (32 production)', 'auc': 0.6924, 'pr_auc': 0.3339, 'f1': 0.36, 'precision': 0.26, 'recall': 0.585, 'p@0.5%': np.float64(0.891), 'p@1%': np.float64(0.781), 'p@5%': np.float64(0.458), 'p@10%': np.float64(0.38), 'carrier_wtd_p@1%': np.float64(0.696)}
Baseline reproduces production (AUC 0.6924, p@1% 0.781) within +/-0.002.


## 9 — Weather models: A (upper bound) and B (honest)

In [10]:
# Each variant's columns are renamed to the plain ORIG_WX_* / DEST_WX_* names, so A and B
# models have identical feature names and the candidate config reads naturally.
ren = lambda sfx, cols: {c + sfx: c for c in cols}
feats_A = PROD_FEATURES + [c + '_A' for c in WX_FEATURES]
feats_B = PROD_FEATURES + [c + '_B' for c in WX_FEATURES]

runs['WEATHER_A'] = run_blend(feats_A, ren('_A', WX_FEATURES), tag='+ WEATHER A (at departure)')
runs['WEATHER_B'] = run_blend(feats_B, ren('_B', WX_FEATURES), tag='+ WEATHER B (3h before)')

evals = [base_eval,
         evaluate('+ WEATHER A (upper bound)', runs['WEATHER_A']['blend']),
         evaluate('+ WEATHER B (honest)',      runs['WEATHER_B']['blend'])]
comp = pd.DataFrame(evals).set_index('config')
delta = (comp - comp.iloc[0]).iloc[1:]
delta.index = [f'delta {i}' for i in delta.index]
print(pd.concat([comp, delta.round(4)]).to_string())

+ WEATHER A (at departure) 52 features | xgb best iter 198 | 118s


+ WEATHER B (3h before)    52 features | xgb best iter 68 | 110s


                                    auc  pr_auc     f1  precision  recall  p@0.5%   p@1%   p@5%  p@10%  carrier_wtd_p@1%
BASELINE (32 production)         0.6924  0.3339  0.360      0.260   0.585   0.891  0.781  0.458  0.380             0.696
+ WEATHER A (upper bound)        0.7002  0.3478  0.366      0.261   0.608   0.874  0.804  0.486  0.395             0.741
+ WEATHER B (honest)             0.6910  0.3274  0.355      0.243   0.657   0.876  0.759  0.445  0.369             0.692
delta + WEATHER A (upper bound)  0.0078  0.0139  0.006      0.001   0.023  -0.017  0.023  0.028  0.015             0.045
delta + WEATHER B (honest)      -0.0014 -0.0065 -0.005     -0.017   0.072  -0.015 -0.022 -0.013 -0.011            -0.004


### 9.1 — The value of a perfect forecast (A − B)

In [11]:
gap = (comp.loc['+ WEATHER A (upper bound)'] - comp.loc['+ WEATHER B (honest)']).round(4)
b_gain = (comp.loc['+ WEATHER B (honest)'] - comp.loc['BASELINE (32 production)']).round(4)
a_gain = (comp.loc['+ WEATHER A (upper bound)'] - comp.loc['BASELINE (32 production)']).round(4)
print(pd.DataFrame({'A - baseline': a_gain, 'B - baseline': b_gain, 'A - B (perfect-forecast value)': gap}).to_string())
share = b_gain['auc'] / a_gain['auc'] if a_gain['auc'] > 0 else np.nan
print(f'\nB captures {100*share:.0f}% of the AUC gain a perfect forecast would give.' if np.isfinite(share)
      else '\nA does not improve AUC, so the share captured by B is undefined.')

                  A - baseline  B - baseline  A - B (perfect-forecast value)
auc                     0.0078       -0.0014                          0.0092
pr_auc                  0.0139       -0.0065                          0.0204
f1                      0.0060       -0.0050                          0.0110
precision               0.0010       -0.0170                          0.0180
recall                  0.0230        0.0720                         -0.0490
p@0.5%                 -0.0170       -0.0150                         -0.0020
p@1%                    0.0230       -0.0220                          0.0450
p@5%                    0.0280       -0.0130                          0.0410
p@10%                   0.0150       -0.0110                          0.0260
carrier_wtd_p@1%        0.0450       -0.0040                          0.0490

B captures -18% of the AUC gain a perfect forecast would give.


## 10 — Variant B: feature importance (XGBoost component, same measure as notebook 2)

In [12]:
mB = runs['WEATHER_B']
imp = (pd.DataFrame({'feature': mB['features'], 'importance': mB['models']['xgb'].feature_importances_})
       .sort_values('importance', ascending=False).reset_index(drop=True))
imp['rank'] = imp.index + 1
imp['is_weather'] = imp['feature'].isin(WX_FEATURES)
print('Top 15:')
print(imp.head(15).to_string(index=False))
print(f'\nWeather features, all {len(WX_FEATURES)} ranked (of {len(imp)}):')
print(imp[imp['is_weather']][['rank', 'feature', 'importance']].to_string(index=False))
print(f'\nTotal importance share of weather features: {imp.loc[imp["is_weather"], "importance"].sum():.3f}')

Top 15:
             feature  importance  rank  is_weather
        LEG_FRACTION    0.085801     1       False
             LEG_NUM    0.079654     2       False
        DEP_HOUR_SIN    0.050952     3       False
SCHEDULED_BUFFER_MIN    0.044903     4       False
     AVAILABLE_SLACK    0.044747     5       False
   ORIG_WX_PRECIP_IN    0.043265     6        True
   DEST_WX_PRECIP_IN    0.039709     7        True
              ORIGIN    0.025677     8       False
  ORIG_WX_IS_THUNDER    0.024525     9        True
      DEST_WX_IS_IFR    0.024225    10        True
  DEST_WX_IS_THUNDER    0.023929    11        True
     DEST_WX_IS_SNOW    0.023455    12        True
     TAIL_LEGS_TODAY    0.023264    13       False
   OP_UNIQUE_CARRIER    0.022360    14       False
            DEP_HOUR    0.022243    15       False

Weather features, all 20 ranked (of 52):
 rank            feature  importance
    6  ORIG_WX_PRECIP_IN    0.043265
    7  DEST_WX_PRECIP_IN    0.039709
    9 ORIG_WX_IS_THUNDE

## 11 — Variant B ablation: origin only vs destination only vs both

In [13]:
runs['B_ORIG'] = run_blend(PROD_FEATURES + [c + '_B' for c in ORIG_WX], ren('_B', ORIG_WX), tag='B origin only')
runs['B_DEST'] = run_blend(PROD_FEATURES + [c + '_B' for c in DEST_WX], ren('_B', DEST_WX), tag='B destination only')
abl = pd.DataFrame([base_eval,
                    evaluate('B origin weather only', runs['B_ORIG']['blend']),
                    evaluate('B destination weather only', runs['B_DEST']['blend']),
                    evaluate('B both', runs['WEATHER_B']['blend'])]).set_index('config')
abl_d = (abl - abl.iloc[0]).iloc[1:].round(4)
abl_d.index = [f'delta {i}' for i in abl_d.index]
print(pd.concat([abl, abl_d]).to_string())

B origin only              42 features | xgb best iter 132 | 119s


B destination only         42 features | xgb best iter 81 | 99s


                                     auc  pr_auc     f1  precision  recall  p@0.5%   p@1%   p@5%  p@10%  carrier_wtd_p@1%
BASELINE (32 production)          0.6924  0.3339  0.360      0.260   0.585   0.891  0.781  0.458  0.380             0.696
B origin weather only             0.6916  0.3298  0.355      0.244   0.651   0.876  0.771  0.452  0.370             0.693
B destination weather only        0.6881  0.3251  0.355      0.249   0.616   0.874  0.761  0.443  0.366             0.692
B both                            0.6910  0.3274  0.355      0.243   0.657   0.876  0.759  0.445  0.369             0.692
delta B origin weather only      -0.0008 -0.0041 -0.005     -0.016   0.066  -0.015 -0.010 -0.006 -0.010            -0.003
delta B destination weather only -0.0043 -0.0088 -0.005     -0.011   0.031  -0.017 -0.020 -0.015 -0.014            -0.004
delta B both                     -0.0014 -0.0065 -0.005     -0.017   0.072  -0.015 -0.022 -0.013 -0.011            -0.004


## 12 — Does weather help more in some seasons?

The test split is the last 15% of dates, which falls entirely inside July 2025, so a by-month
AUC cannot be computed. As the closest available substitute, the test set is split by the
weather the honest model could actually see (Variant B origin conditions).

In [14]:
print(f'Test months: {sorted(test["FL_DATE"].dt.month.unique())}  '
      f'({test["FL_DATE"].min().date()} to {test["FL_DATE"].max().date()})')
print('Test split spans a single month - AUC by month is not possible.\n')

tb = test.reset_index(drop=True)
segments = {
    'all test flights': np.ones(len(tb), bool),
    'origin thunder 3h before': tb['ORIG_WX_IS_THUNDER_B'].eq(1).values,
    'origin IFR 3h before':     tb['ORIG_WX_IS_IFR_B'].eq(1).values,
    'origin precip > 0 3h before': tb['ORIG_WX_PRECIP_IN_B'].gt(0).values,
    'dest thunder 3h before':   tb['DEST_WX_IS_THUNDER_B'].eq(1).values,
    'no adverse weather (VFR, no TS/precip either end)':
        (tb['ORIG_WX_IS_IFR_B'].eq(0) & tb['ORIG_WX_IS_THUNDER_B'].eq(0) & tb['ORIG_WX_PRECIP_IN_B'].fillna(0).eq(0)
         & tb['DEST_WX_IS_IFR_B'].eq(0) & tb['DEST_WX_IS_THUNDER_B'].eq(0)).values,
}
rows = []
for nm, m in segments.items():
    yt = y_test.values[m]
    if m.sum() < 500 or yt.min() == yt.max():
        continue
    a0 = roc_auc_score(yt, runs['BASELINE']['blend'][m])
    aB = roc_auc_score(yt, runs['WEATHER_B']['blend'][m])
    rows.append({'segment': nm, 'flights': int(m.sum()), 'delay_rate': round(yt.mean(), 3),
                 'auc_baseline': round(a0, 4), 'auc_B': round(aB, 4), 'delta': round(aB - a0, 4)})
seg = pd.DataFrame(rows)
print(seg.to_string(index=False))

Test months: [np.int32(7)]  (2025-07-19 to 2025-07-31)
Test split spans a single month - AUC by month is not possible.



                                          segment  flights  delay_rate  auc_baseline  auc_B   delta
                                 all test flights   223101       0.156        0.6924 0.6910 -0.0014
                         origin thunder 3h before     2373       0.319        0.6873 0.6924  0.0051
                             origin IFR 3h before     6048       0.153        0.7064 0.7097  0.0033
                      origin precip > 0 3h before     9826       0.232        0.7030 0.7099  0.0069
                           dest thunder 3h before     2554       0.279        0.6774 0.6800  0.0027
no adverse weather (VFR, no TS/precip either end)   198889       0.151        0.6895 0.6852 -0.0043


## 13 — Save the Variant B candidate

Written to `results/candidates/weather_b/` only. Production artifacts in `results/` are not
touched. The blend is the unweighted mean of the three components' probabilities, as in
production. HGB and LightGBM receive categoricals as integer codes from the training-set
category levels stored in `config.json`, exactly as notebook 2 feeds them.

In [15]:
CAND_DIR.mkdir(parents=True, exist_ok=True)
assert CAND_DIR.resolve() != Path('results').resolve()

mods = mB['models']
mods['xgb'].save_model(str(CAND_DIR / 'xgb.json'))
mods['lgb'].booster_.save_model(str(CAND_DIR / 'lgbm.txt'))
joblib.dump(mods['hgb'], CAND_DIR / 'hgb.joblib')

b_row = comp.loc['+ WEATHER B (honest)'].to_dict()
config = {
    'name': 'Layer 3 primary-delay blend + Variant B weather (3h before departure)',
    'status': 'CANDIDATE - not promoted',
    'source_notebook': 'fdpis_7_weather.ipynb',
    'features': mB['features'],
    'n_features': len(mB['features']),
    'production_features': PROD_FEATURES,
    'weather_features': WX_FEATURES,
    'cat_cols': CAT_COLS,
    'category_levels': {c: [str(v) for v in cat_types[c].categories] for c in CAT_COLS},
    'categorical_handling': {'xgb': 'pandas category dtype with category_levels (enable_categorical)',
                             'hgb': 'integer codes of category_levels (-1 = unseen/NaN)',
                             'lgbm': 'integer codes of category_levels (-1 = unseen/NaN)'},
    'blend': 'mean of predict_proba[:,1] from xgb, hgb, lgbm',
    'components': {'xgb': 'xgb.json', 'hgb': 'hgb.joblib', 'lgbm': 'lgbm.txt'},
    'weather_definition': {
        'source': 'Iowa Environmental Mesonet ASOS archive (METAR), routine + special reports',
        'lookup_hour_utc': 'floor(scheduled departure, UTC) - 3h, at BOTH origin and destination',
        'observation': 'latest METAR at or before that hour (within 59 min), forward-filled <= 2 h',
        'guarantee': 'every observation used is >= 3h before scheduled departure (asserted per flight)',
        'ceiling': 'lowest BKN/OVC/VV layer, 10000 if none, capped at 10000',
        'visibility_cap_mi': 10, 'trace_precip_in': 0.0001,
    },
    'test_metrics': {k: float(v) for k, v in b_row.items()},
    'baseline_test_metrics': {k: float(v) for k, v in comp.loc['BASELINE (32 production)'].to_dict().items()},
    'split': {'train_end': str(pd.Timestamp(train_end).date()), 'val_end': str(pd.Timestamp(val_end).date())},
    'scale_pos_weight': float(scale_pos_weight),
    'xgb_best_iteration': int(mods['xgb'].best_iteration),
}
(CAND_DIR / 'config.json').write_text(json.dumps(config, indent=2))
pd.concat([comp, delta.round(4)]).to_csv(CAND_DIR / 'comparison.csv')
pd.concat([abl, abl_d]).to_csv(CAND_DIR / 'ablation_origin_dest.csv')
imp.to_csv(CAND_DIR / 'feature_importance.csv', index=False)
seg.to_csv(CAND_DIR / 'weather_segments.csv', index=False)

# Round-trip check: reloaded components reproduce the in-memory blend.
X = prep(test, feats_B, ren('_B', WX_FEATURES))
x2 = xgb.XGBClassifier(); x2.load_model(str(CAND_DIR / 'xgb.json'))
l2 = lgb.Booster(model_file=str(CAND_DIR / 'lgbm.txt'))
h2 = joblib.load(CAND_DIR / 'hgb.joblib')
re_blend = np.mean([x2.predict_proba(X)[:, 1], h2.predict_proba(to_numeric(X))[:, 1],
                    l2.predict(to_numeric(X))], axis=0)
print(f'Reload max abs diff vs in-memory blend: {np.abs(re_blend - mB["blend"]).max():.2e}')
print('Saved:', sorted(p.name for p in CAND_DIR.iterdir()))

Reload max abs diff vs in-memory blend: 0.00e+00
Saved: ['ablation_origin_dest.csv', 'comparison.csv', 'config.json', 'feature_importance.csv', 'hgb.joblib', 'lgbm.txt', 'weather_segments.csv', 'xgb.json']


## 14 — Promotion rule

In [16]:
d_auc = b_gain['auc']
d_p1  = b_gain['p@1%']
promote = (d_auc >= 0.005) and (d_p1 >= -0.01)
print(f'Variant B vs baseline: delta AUC {d_auc:+.4f} (needs >= +0.005), '
      f'delta p@1% {d_p1:+.3f} (must be >= -0.010)')
print('RECOMMENDATION:', 'PROMOTE Variant B' if promote else 'KEEP as documented experiment - do not promote')

Variant B vs baseline: delta AUC -0.0014 (needs >= +0.005), delta p@1% -0.022 (must be >= -0.010)
RECOMMENDATION: KEEP as documented experiment - do not promote
